Plan:
1. Prepare 61 images for visual review: the original ultrasound image on the left and the ultrasound image with the BiomedParse segmentation mask overlay on the right.
2. Create a review table with one evaluation criterion, overall_visual_assessment (Good, Partial, Poor, Unclear), and a reviewer_notes field.
3. Conduct a manual visual assessment and compare the distribution of ratings with the Gemini results.

In [ ]:
# Import libraries and mount Google Drive.

from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image

from google.colab import drive
drive.mount("/content/drive")

# Define project directories.
PROJECT_DIR = Path("/content/drive/MyDrive/ovarian_ultrasound_vlm")

DATASET_DIR = Path(
    "/content/drive/MyDrive/biomedparse_project/"
    "nnunet_raw/Dataset001_Follicles"
)

IMAGES_DIR = DATASET_DIR / "imagesTr"
MASKS_DIR = DATASET_DIR / "labelsTr"

MANIFEST_PATH = PROJECT_DIR / "image_manifest.csv"

OUTPUT_DIR = PROJECT_DIR / "evaluation" / "biomedparse_qualitative"
FIGURES_DIR = PROJECT_DIR / "figures" / "biomedparse_qualitative"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Manifest exists:", MANIFEST_PATH.exists())
print("Images directory exists:", IMAGES_DIR.exists())
print("Masks directory exists:", MASKS_DIR.exists())

This notebook evaluates the visual plausibility of BiomedParse pseudo-labels on the same 61 validation images used in the Gemini experiment

In [ ]:
# Load the manifest and select validation images.

manifest = pd.read_csv(MANIFEST_PATH)
validation = manifest.loc[manifest["split"] == "val"].copy()

print("Validation images:", len(validation))
print(validation["category"].value_counts())

assert len(validation) == 61
assert validation["case_id"].is_unique

# Verify image and pseudo-label paths.
missing_images = []
missing_masks = []

for row in validation.itertuples():
    if not Path(row.image_path).exists():
        missing_images.append(row.case_id)

    if not Path(row.pseudo_label_path).exists():
        missing_masks.append(row.case_id)

print("Missing images:", len(missing_images))
print("Missing masks:", len(missing_masks))

assert not missing_images, missing_images
assert not missing_masks, missing_masks

print("All 61 image-mask pairs are available.")

Use the same validation split as the Gemini evaluation to ensure a consistent comparison